# Practical Exercise: Exploring an Agricultural Knowledge Base with Python

## Objective
In this practical, you will use a small agricultural question-and-answer dataset to understand the basic idea of **retrieval**.

You will:

1. Load the Excel dataset.
2. Keep only the English questions and answers.
3. Explore a few examples.
4. Type a new agricultural question.
5. Ask Python to find the most similar question in the knowledge base.
6. Retrieve the corresponding answer.
7. Experiment with different ways of asking the same question.

### Key idea

**Your question → Search the knowledge base → Find relevant information → Return an answer**

This is a simplified demonstration of the **retrieval** part of Retrieval-Augmented Generation (RAG).

> You are **not training an AI model** in this exercise.


## Step 1 — Import the required Python libraries

We will use:

- `pandas` to read the Excel file
- `scikit-learn` to compare questions


In [ ]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Load the dataset

Make sure the Excel file **Mkulima Knowledge Base_SwEN(1).xlsx** is in the same folder as this notebook.

Run the cell below.


In [ ]:
file_name = "/content/drive/MyDrive/Datasets/Mkulima Knowledge_Base.xlsx"

df = pd.read_excel(file_name)

print("Dataset loaded successfully.")
print("Number of rows:", len(df))

df.head()

Dataset loaded successfully.
Number of rows: 174


,Maswali,Qestions,Majibu,Unnamed: 3,Unnamed: 4,Answers
0,Jinsi gani upepo unavyosababisha mmomonyoko wa...,How does wind cause soil erosion and affect co...,Upepo huondoa safu ya juu ya udongo na kupelek...,NaN,NaN,Wind removes the top layer of soil and affects...
1,Umuhimu wa uchavushaji kwa mmea wa mahindi ni ...,What is the importance of pollination for corn...,Uchavushaji husaidia mmea kuweza kuzalisha tun...,NaN,NaN,Pollination helps the plant to produce fruit. ...
2,Yapi kati ya Mahindi yaliyokauka na mahindi ma...,What are the differences between dried and fre...,Bei ya mahindi mabichi au makavu hutegemea sok...,NaN,NaN,The price of fresh or dry corn depends on the ...
3,Mmea wa mahindi una aina gani ya mizizi?,What type of root system does a corn plant have?,Mmea wa mahindi huwa na aina ya mizizi miemaba...,NaN,NaN,"Corn plants have a type of thin, fibrous roots..."
4,Jinsi gani naweza kuoboresha virutubisho kati...,How can I improve the nutrients in corn plants?,Kuna namna mbalimbali za kuboresha virutisho k...,NaN,NaN,There are various ways to improve nutrients in...


## Step 3 — Keep only the English questions and answers

The original dataset contains both Swahili and English.

For this exercise, we will use:

- `Qestions` = English questions
- `Answers` = English answers

We will rename them to make the dataset easier to read.


In [ ]:
data = df[["Qestions", "Answers"]].copy()

data.columns = ["Question", "Answer"]

data = data.dropna()

print("Number of English question-answer pairs:", len(data))

data.head()

Number of English question-answer pairs: 172


,Question,Answer
0,How does wind cause soil erosion and affect co...,Wind removes the top layer of soil and affects...
1,What is the importance of pollination for corn...,Pollination helps the plant to produce fruit. ...
2,What are the differences between dried and fre...,The price of fresh or dry corn depends on the ...
3,What type of root system does a corn plant have?,"Corn plants have a type of thin, fibrous roots..."
4,How can I improve the nutrients in corn plants?,There are various ways to improve nutrients in...


## Step 4 — Look at a few examples

Run the cell below to see five random question-and-answer pairs from the agricultural knowledge base.

Run it more than once and observe how the examples change.


In [ ]:
data.sample(5)

,Question,Answer
99,What causes early stem cracks in corn?,Insects that bore into the maize stem.
111,How can I use natural fertilizers in corn?,By mixing into the soil after loosening it or ...
72,How often should I apply pesticides in the cor...,Weeding maize depends on weather conditions an...
4,How can I improve the nutrients in corn plants?,There are various ways to improve nutrients in...
173,What precautions should I consider when storin...,"Ensuring that maize is properly dried, clean, ..."


## Step 5 — Prepare the questions for searching

We will use a simple method called **TF-IDF**.

You do not need to understand the mathematics for this exercise.

The important idea is:

> Python converts the questions into a form that allows it to compare how similar their words are.


In [ ]:
questions = data["Question"].fillna("")

vectorizer = TfidfVectorizer(stop_words="english")

question_vectors = vectorizer.fit_transform(questions)

print("The knowledge base is ready for searching.")


The knowledge base is ready for searching.


## Step 6 — Create a simple search function

The function below will:

1. Take your question.
2. Compare it with all questions in the knowledge base.
3. Find the closest question.
4. Return its answer.

Run the cell once to create the function.


In [ ]:
def search_knowledge_base_customised(user_question):
    print("YOUR QUESTION")
    print(user_question)
    # Foundation model code
    """
    Vectorises the user query, compares against dataset embeddings via cosine similarity,
    and retrieves the top matching Q&A pairs from your pandas DataFrame.
    """
    user_query = user_question
    top_k=2
    query_embedding = embedder.encode(user_query, convert_to_tensor=True)
    similarity_scores = util.cos_sim(query_embedding, dataset_embeddings)[0]

    top_results = torch.topk(similarity_scores, k=top_k)

    retrieved_contexts = []
    for idx in top_results.indices:
        q_text = data.iloc[idx.item()]["Question"]
        a_text = data.iloc[idx.item()]["Answer"]
        retrieved_contexts.append(f"Q: {q_text}\nA: {a_text}")

    retrieved_info = "\n\n".join(retrieved_contexts)

    # 3. Construct prompt for FLAN-T5
    prompt = f"""Answer the user's agricultural question using only the context retrieved from the database below.

Retrieved Context:
{retrieved_info}

User Question: {user_query}
Final Answer:"""

    # 4. Pass to LLM (FLAN-T5)
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = generator.generate(**inputs, max_new_tokens=150, temperature=0.2)

    # 5. Return final answer text
    final_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    user_vector = vectorizer.transform([user_question])

    similarities = cosine_similarity(
        user_vector,
        question_vectors
    )[0]

    best_match = similarities.argmax()

    matched_question = data.iloc[best_match]["Question"]
    matched_answer = data.iloc[best_match]["Answer"]
    score = similarities[best_match]
    print("\nRETRIEVED ANSWER WITHOUT FOUNDATION MODEL")
    print(matched_answer)
    print("\nRETRIEVED ANSWER WITH FOUNDATION MODEL")
    print(final_answer)


## Step 7 — Try your first question

Run the example below.


## Step 8 — Your turn

Change the question inside the quotation marks and run the cell again.

Try questions such as:

- How can I protect maize from pests?
- Why are my maize plants turning yellow?
- How should I store maize after harvest?
- What causes poor maize growth?

You can also write your own question.


## Step 9 — Ask the same question in different ways

Now test whether changing the wording changes the result.

Run each example below one at a time.


## Discussion

Discuss the following questions with your group:

1. Did the three differently worded questions retrieve the same information?
2. Which question produced the best match?
3. What happened when you used words that were very different from those in the dataset?
4. What are the limitations of this simple word-based search method?
5. Why might a trusted agricultural knowledge base be useful when building an AI advisory system?

### Connection to RAG

In this exercise, we only performed retrieval:

**Question → Search → Retrieve information**

In a full RAG system, the retrieved information would then be given to a language model:

**Question → Search → Retrieve information → LLM → Final answer**


In [ ]:
# Step 1: Install required libraries
!pip install transformers torch pandas sentence-transformers


In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [ ]:
# Step 2: Load your pandas dataset
# Adjust filename to match your local file or Colab upload path
# df = pd.read_csv("agricultural_data.csv")

# Step 3: Load the embedding model and FLAN-T5-Base
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Pre-vectorise the 'Question' column from your dataset
dataset_embeddings = embedder.encode(data["Question"].astype(str).tolist(), convert_to_tensor=True)

# Load FLAN-T5-Base
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
search_knowledge_base_customised ("can corn adjust the soil nitrogen level?")

YOUR QUESTION
can corn adjust the soil nitrogen level?

RETRIEVED ANSWER WITHOUT FOUNDATION MODEL
No. Corn plants are not a type of plant that can fix the level of nitrogen in the soil.

RETRIEVED ANSWER WITH FOUNDATION MODEL
No


In [ ]:
Test_Q1 = "Can corn adjust the soil Nitrogen level?"
Test_Q2 = "What is the influence of soil Nitrogen to corn growth?"
search_knowledge_base_customised (Test_Q2)

YOUR QUESTION
What is the influence of soil Nitrogen to corn growth?

RETRIEVED ANSWER WITHOUT FOUNDATION MODEL
No. Corn plants are not a type of plant that can fix the level of nitrogen in the soil.

RETRIEVED ANSWER WITH FOUNDATION MODEL
The level of nitrogen in the soil is a factor in the growth of corn.
